# 서브 워드 비교 실험

서브워드 라이브러리 HuggingFace Tokenizer와 SentencePiece를 비교하는 과제 노트북입니다.

진행 순서는 다음과 같습니다.

1. 데이터 준비 : 네이버 영화 리뷰 [[링크]](https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt)
2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정
3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석
4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정
5. 기타 추가 실험
6. 사용성과 장단점 분석

## 1. 데이터 준비 : 네이버 영화 리뷰

### HuggingFace Tokenizer와 SentencePiece 설치하기

In [ ]:
!pip install -q sentencepiece tokenizers

### 네이버 리뷰 데이터 불러오기

In [ ]:
import requests

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
response = requests.get(url)

with open("corpus.txt", "wb") as f:
    f.write(response.content)

In [ ]:
import os

file_path = "/content/corpus.txt"

if os.path.exists(file_path):
    print("다운로드 완료")
else:
    print("파일을 찾을 수 없음")

다운로드 완료


### 데이터 정제하기
현재 데이터에는 `id`, `document`, `label`이 있습니다. 이 중 실질적인 문장 데이터인 `document`만 활용합니다.

In [ ]:
# [[YOUR CODE]]
# 데이터 정제
import pandas as pd

# 데이터 읽기
df = pd.read_csv(file_path, sep='\t')

# document 컬럼만 추출
documents = df['document']

# 결측치 제거
documents = documents.dropna()

# 문자열로 변환
documents = documents.astype(str)

# 문장만 저장
with open(file_path, "w", encoding="utf-8") as f:
    for sentence in documents:
        f.write(sentence.strip() + "\n")

print("정제 완료")
print("문장 수:", len(documents))

정제 완료
문장 수: 149995


## 2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정

HuggingFace BPE는 SentencePiece와 내제된 철학이 다릅니다.  
고전적인 BPE는 단어 단위 알고리즘에서 출발하였으며 단어 목록을 입력으로 받아 문장 분리 -> 단어 분리 -> BPE를 가정합니다.  
때문에 이 철학을 따르는 HuggingFace BPE는 `Witespace()` 혹은 `ByteLevel()`과 같은 pre-tokenizer가 필요합니다.  
반면 SentencePiece는 공백도 학습하자는 철학으로 공백에 대하여 \_\_로 처리하여 기본 문장에서 공백을 기준으로 나누는 것은 동일하지만 문장 전체를 입력받게 되며 다음과 같이 처리합니다.  

"오늘은 날이 참 밝다."  
"오늘은\_\_날이\_\_참\_\_밝다."

만일 SentencePiece의 철학과 유사하게 진행하려면 `Metqaspace()`를 활용하십시오.

### Sentencepiece를 사용하여 단어사전 구축

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input=file_path,
    model_prefix="spm",
    vocab_size=2000,
    model_type="bpe"
)

sp = spm.SentencePieceProcessor()
sp.load("spm.model")

sentence = input("문장을 입력하세요: ")

print(sp.encode(sentence, out_type=str))

문장을 입력하세요: 오늘 날이 참 맑다
['▁오', '늘', '▁', '날', '이', '▁참', '▁', '맑', '다']


### HuggingFace Tokenizer를 사용하여 단어사전 구축

> 인용구 추가



In [ ]:
# [[YOUR CODE]]
# HuggingFace Tokenizer를 사용한 단어사전 구축

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Metaspace()

trainer = BpeTrainer(
    vocab_size=1000,
    special_tokens=["[UNK]"]
)

tokenizer.train(
    files=[file_path],
    trainer=trainer

)
sentence = input("문장을 입력하세요: ")

encoding = tokenizer.encode(sentence)

print("\n토큰:")
print(encoding.tokens)

문장을 입력하세요: 오늘 날이 참 맑다

토큰:
['▁', '오', '늘', '▁', '날', '이', '▁', '참', '▁', '맑', '다']


## 3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석

### 상위 30개 기준, 단어사전 조회하기

In [ ]:
import sentencepiece as spm
from tokenizers import Tokenizer

# SentencePiece
sp = spm.SentencePieceProcessor()
sp.load("spm.model")

print("===== SentencePiece TOP 30 =====")

for i in range(30):
    print(i, sp.id_to_piece(i))


from collections import Counter

counter = Counter()

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        tokens = tokenizer.encode(line.strip()).tokens
        counter.update(tokens)

print("===== HuggingFace Token Frequency TOP 30 =====")

for token, freq in counter.most_common(30):
    print(token, freq)

===== SentencePiece TOP 30 =====
0 <unk>
1 <s>
2 </s>
3 ..
4 영화
5 ▁영화
6 ▁이
7 ▁아
8 ...
9 ᄏᄏ
10 ▁그
11 ▁보
12 ▁재
13 !!
14 니다
15 ▁정
16 ▁나
17 는데
18 너무
19 ▁다
20 ▁하
21 ▁좋
22 ▁없
23 ▁어
24 ▁너무
25 ▁사
26 ▁내
27 ▁감
28 ▁진
29 ▁재미
===== HuggingFace Token Frequency TOP 30 =====
▁ 1137736
. 241461
이 137121
다 110236
는 90570
고 70831
지 66721
화 65575
영 64734
가 57022
하 56962
도 50165
아 47071
에 45283
나 42976
기 42464
한 41010
보 37967
어 37906
의 37900
만 37106
ㅋ 35257
은 34237
리 33246
로 32595
게 32224
을 32042
라 31896
! 31840
서 31685


### 분절된 토큰 빈도 조회

In [ ]:
# [[YOUR CODE]]
import sentencepiece as spm
from collections import Counter
from tokenizers import Tokenizer
from collections import Counter
sp = spm.SentencePieceProcessor()
sp.load("spm.model")

counter = Counter()

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        tokens = sp.encode(line.strip(), out_type=str)
        counter.update(tokens)

print("===== SentencePiece Token Frequency (Top 30) =====")
for token, freq in counter.most_common(30):
    print(token, freq)

tokenizer = Tokenizer.from_file("hf_bpe.json")

counter = Counter()

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        encoding = tokenizer.encode(line.strip())
        counter.update(encoding.tokens)

print("===== HuggingFace Token Frequency (Top 30) =====")
for token, freq in counter.most_common(30):
    print(token, freq)

===== SentencePiece Token Frequency (Top 30) =====
▁ 317089
이 78269
. 70255
다 50253
도 47947
는 47334
가 41355
고 37152
에 35175
지 35115
▁영화 34683
을 32042
.. 31804
의 30693
한 29954
은 27291
나 25939
... 25092
리 24647
▁이 24123
게 20892
▁아 20551
기 20223
, 18960
만 18787
어 18654
아 18252
▁그 17868
라 17610
로 17138


Exception: No such file or directory (os error 2)

## 4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정

### 단일 문장 변환 속도 측정

In [ ]:
import time
import sentencepiece as spm
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# =========================
# 모델 로드
# =========================

sp = spm.SentencePieceProcessor()
sp.load("spm.model")

hf_tokenizer = Tokenizer.from_file("hf_bpe.json")


# =========================
# 1. 학습 속도
# =========================

print("\n===== 학습 속도 =====")

start = time.time()

spm.SentencePieceTrainer.train(
    input=file_path,
    model_prefix="spm_test",
    vocab_size=8000,
    model_type="bpe"
)

sp_train_time = time.time() - start
print("SentencePiece 학습 시간:", sp_train_time)


hf = Tokenizer(models.BPE())
hf.pre_tokenizer = pre_tokenizers.Metaspace()

trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["[UNK]"]
)

start = time.time()
hf.train([file_path], trainer)
hf_train_time = time.time() - start

print("HuggingFace 학습 시간:", hf_train_time)


# =========================
# 2. 단일 문장 변환 속도
# =========================

sentence = input("\n문장 입력: ")
n = int(input("반복 횟수: "))

print("\n===== 단일 문장 변환 속도 =====")

start = time.time()
for _ in range(n):
    sp.encode(sentence, out_type=str)
sp_time = time.time() - start

start = time.time()
for _ in range(n):
    hf_tokenizer.encode(sentence)
hf_time = time.time() - start

print("SentencePiece:", sp_time)
print("HuggingFace:", hf_time)

# =========================
# 3. 코퍼스 변환 속도
# =========================

print("\n===== 코퍼스 변환 속도 =====")

start = time.time()
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        sp.encode(line.strip(), out_type=str)
sp_corpus_time = time.time() - start

start = time.time()
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        hf_tokenizer.encode(line.strip())
hf_corpus_time = time.time() - start

print("SentencePiece 코퍼스:", sp_corpus_time)
print("HuggingFace 코퍼스:", hf_corpus_time)


# =========================
# 4. 최종 비교 (핵심 추가 부분)
# =========================

print("\n===== 최종 비교 (차이 분석) =====")
print("코퍼스 SP:", sp_corpus_time)
print("코퍼스 HF:", hf_corpus_time)

print("차이:", hf_corpus_time - sp_corpus_time)
print("비율:", hf_corpus_time / sp_corpus_time)

### 코퍼스 전체 변환 속도 측정

In [ ]:
# [[YOUR CODE]]
import time
import sentencepiece as spm
from tokenizers import Tokenizer

sp = spm.SentencePieceProcessor()
sp.load("spm.model")

start = time.time()

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        sp.encode(line.strip(), out_type=str)

end = time.time()

print("SentencePiece 코퍼스 변환 시간:", end - start)


tokenizer = Tokenizer.from_file("hf_bpe.json")

start = time.time()

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        tokenizer.encode(line.strip())

end = time.time()

print("HuggingFace 코퍼스 변환 시간:", end - start)

## 5. 기타 추가 실험

In [ ]:
# [[YOUR CODE]]

## 6. 사용성과 장단점 분석

**장점**  
setencepiece
장점: 언어 독립적
     공백까지 학습
     재현성이 높음
   

HuggingFace Tokenizer
장점
   속도가 빠름
   다양한 tokenizer 구조지원

**단점**

setencepiece
단점: 학습시 비교적 느림
HuggingFace Tokenizer

단점: pre-tokenizer 영향 큼 → 결과 흔들릴 수 있음
SentencePiece보다 일관성 떨어질 수 있음

설정에 따라 결과 차이 큼